# 🌋 Seismic Intelligence — Exploratory Data Analysis
Deep-dive into earthquake patterns, Gutenberg-Richter law, and ML baselines.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta

from api.usgs_fetch import fetch_live_feed, fetch_historical
from data.preprocess import clean_raw, engineer_features
from analytics.risk_zones import compute_risk_grid, zone_risk_summary
from analytics.aftershock import full_aftershock_report, expected_aftershocks
from utils.helpers import compute_global_stats, magnitude_label

plt.style.use('dark_background')
print('✅ Imports OK')

In [ ]:
# Load data
raw = fetch_live_feed('2.5_month')
df = engineer_features(clean_raw(raw))
print(f'Loaded {len(df)} earthquakes | Columns: {list(df.columns)}')
df.head()

In [ ]:
# Global stats
stats = compute_global_stats(df)
for k,v in stats.items(): print(f'  {k:25s}: {v}')

In [ ]:
# Gutenberg-Richter frequency-magnitude distribution
bins = np.arange(2, 10, 0.2)
counts, edges = np.histogram(df['magnitude'], bins=bins)
centers = 0.5*(edges[:-1]+edges[1:])

# Fit b-value
mc = 3.0  # completeness magnitude
above = df[df['magnitude'] >= mc]['magnitude']
b = np.log10(np.e) / (above.mean() - mc + 1e-9)

fig, ax = plt.subplots(figsize=(10,5), facecolor='#0d0d0d')
ax.set_facecolor('#0d0d0d')
log_c = np.log10(counts + 1)
ax.bar(centers, log_c, width=0.18, color='#f97316', alpha=0.8, label='Observed')
m_fit = np.linspace(mc, 8, 100)
gr_fit = log_c.max() - b*(m_fit - mc)
ax.plot(m_fit, gr_fit, '--', color='#0ea5e9', lw=2, label=f'G-R fit (b={b:.2f})')
ax.set_xlabel('Magnitude', color='#888'); ax.set_ylabel('log₁₀(N)', color='#888')
ax.set_title(f'Gutenberg-Richter Law  |  b-value = {b:.3f}', color='#ddd', pad=12)
ax.legend(framealpha=0); ax.tick_params(colors='#666')
plt.tight_layout(); plt.show()

In [ ]:
# Interactive global map
fig = px.scatter_geo(
    df.sample(min(2000,len(df))),
    lat='latitude', lon='longitude',
    color='magnitude', size='magnitude',
    hover_name='place',
    color_continuous_scale='Inferno',
    projection='natural earth',
    title='Global Earthquake Distribution'
)
fig.update_geos(showland=True, landcolor='#111', oceancolor='#0a1628',
                showocean=True, showcoastlines=True, coastlinecolor='#333')
fig.update_layout(paper_bgcolor='#0d0d0d', font_color='#ccc', height=500)
fig.show()

In [ ]:
# Aftershock analysis for different mainshock magnitudes
magnitudes = [6.0, 7.0, 7.5, 8.0, 8.5]
fig, axes = plt.subplots(1, len(magnitudes), figsize=(18,4), facecolor='#0d0d0d')
for ax, M in zip(axes, magnitudes):
    ax.set_facecolor('#0d0d0d')
    fc = expected_aftershocks(M, min_mag=2.0, days=30)
    ax.bar(range(1,31), fc['daily_counts'], color='#f97316', alpha=0.8)
    ax.set_title(f'M{M}', color='#ddd'); ax.set_xlabel('Days', color='#777')
    ax.tick_params(colors='#555')
    ax.set_ylabel('Expected M2+', color='#777') if M==6.0 else None
plt.suptitle('Omori-Utsu Aftershock Decay by Mainshock Magnitude', color='#ccc', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Risk zones
risk = compute_risk_grid(df)
print(f'Max risk score: {risk["max_risk"]:.1f}')
print(f'Top hotspots:')
print(risk['hotspots'].head(10).to_string(index=False))